# Análisis Exploratorio de Datos (EDA)
## Base de Trabajo de Grado — Empleados Colombia (Permanentes)

Este notebook realiza un análisis exploratorio exhaustivo sobre la base de datos de empleados. El objetivo es comprender la estructura, distribución, calidad y relaciones entre las variables del dataset, con foco en la rotación de personal.

**Secciones:**
1. Carga de datos y librerías
2. Inspección general del dataset
3. Análisis de calidad de datos (nulos, duplicados, tipos)
4. Ingeniería de variables derivadas
5. Análisis univariado — Variables categóricas
6. Análisis univariado — Variables numéricas
7. Análisis bivariado
8. Análisis de correlaciones
9. Análisis temporal (antigüedad y tendencias)
10. Análisis de rotación (turnover)
11. Resumen de hallazgos clave

---
## 1. Carga de datos y librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Librerías cargadas correctamente.')

In [ ]:
df = pd.read_csv('Base_Trabajo_Grado_Adj(Colombia Permanent).csv')
print(f'Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head(10)

---
## 2. Inspección general del dataset

In [ ]:
print('='*60)
print('INFORMACIÓN GENERAL DEL DATASET')
print('='*60)
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Memoria en uso: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print()
print('Columnas y tipos de datos:')
print('-'*60)
df.info()

In [ ]:
print('Estadísticas descriptivas — Variables numéricas:')
df.describe()

In [ ]:
print('Estadísticas descriptivas — Variables categóricas:')
df.describe(include='object')

In [ ]:
# Primeras y últimas filas para verificar consistencia
print('Primeras 5 filas:')
display(df.head())
print('\nÚltimas 5 filas:')
display(df.tail())

---
## 3. Análisis de calidad de datos

In [ ]:
# Análisis de valores nulos
null_analysis = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df) * 100).round(2),
    'No Nulos': df.notnull().sum(),
    'Tipo': df.dtypes
})
null_analysis = null_analysis.sort_values('% Nulos', ascending=False)
print('ANÁLISIS DE VALORES NULOS')
print('='*60)
null_analysis

In [ ]:
# Visualización de nulidad
null_cols = null_analysis[null_analysis['% Nulos'] > 0]
if len(null_cols) > 0:
    fig, ax = plt.subplots(figsize=(12, max(4, len(null_cols)*0.5)))
    bars = ax.barh(null_cols.index, null_cols['% Nulos'], color=sns.color_palette('Reds_r', len(null_cols)))
    ax.set_xlabel('% de valores nulos')
    ax.set_title('Porcentaje de valores nulos por columna')
    for bar, val in zip(bars, null_cols['% Nulos']):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f'{val}%', va='center')
    plt.tight_layout()
    plt.show()
else:
    print('No se encontraron valores nulos estándar en el dataset.')

In [ ]:
# Detección de valores 'nan' como texto y otros placeholders
print('VALORES ESPECIALES (nan como texto, vacíos, placeholders)')
print('='*60)
for col in df.columns:
    if df[col].dtype == 'object':
        nan_text = df[col].isin(['nan', 'NaN', 'NULL', 'null', '', ' ', 'N/A', 'n/a']).sum()
        if nan_text > 0:
            print(f'{col}: {nan_text} valores especiales ({nan_text/len(df)*100:.1f}%)')

# Detección de fechas inválidas
print('\nFECHAS POTENCIALMENTE INVÁLIDAS:')
print('-'*60)
for col in ['Hire Date', 'Termination Date']:
    if col in df.columns:
        invalid = df[df[col].astype(str).str.contains('1900', na=False)][col].nunique()
        count = df[df[col].astype(str).str.contains('1900', na=False)].shape[0]
        print(f'{col}: {count} registros con fecha 1900 (posible placeholder para activos)')

In [ ]:
# Duplicados
print('ANÁLISIS DE DUPLICADOS')
print('='*60)
dup_total = df.duplicated().sum()
dup_emp = df['Employee Code'].duplicated().sum()
print(f'Filas completamente duplicadas: {dup_total}')
print(f'Employee Code duplicados: {dup_emp}')
print(f'Employee Code únicos: {df["Employee Code"].nunique()}')

if dup_emp > 0:
    print(f'\nEmpleados con más de un registro:')
    display(df[df['Employee Code'].duplicated(keep=False)].sort_values('Employee Code').head(10))

In [ ]:
# Cardinalidad de cada columna
cardinality = pd.DataFrame({
    'Valores únicos': df.nunique(),
    '% Cardinalidad': (df.nunique() / len(df) * 100).round(2),
    'Tipo': df.dtypes
}).sort_values('Valores únicos', ascending=False)

print('CARDINALIDAD POR COLUMNA')
print('='*60)
cardinality

---
## 4. Ingeniería de variables derivadas

In [ ]:
# Convertir fechas
df['Hire Date'] = pd.to_datetime(df['Hire Date'], errors='coerce')
df['Termination Date'] = pd.to_datetime(df['Termination Date'], errors='coerce')

# Limpiar 'Termination Code' y 'Termination Reason': reemplazar 'nan' string
for col in ['Termination Code', 'Termination Reason', 'Termination Type']:
    df[col] = df[col].replace('nan', np.nan)

# Variable de rotación (target): el empleado dejó la empresa
df['Turnover'] = df['Active Employee'].map({'N': 1, 'Y': 0})

# Antigüedad en días y años
reference_date = pd.Timestamp('2024-01-01')  # fecha de referencia
df['Tenure_Days'] = np.where(
    df['Turnover'] == 1,
    (df['Termination Date'] - df['Hire Date']).dt.days,
    (reference_date - df['Hire Date']).dt.days
)
df['Tenure_Years'] = (df['Tenure_Days'] / 365.25).round(2)

# Año de contratación
df['Hire_Year'] = df['Hire Date'].dt.year

# Año de terminación (solo para los que se fueron)
df['Termination_Year'] = df['Termination Date'].dt.year
df.loc[df['Turnover'] == 0, 'Termination_Year'] = np.nan

# Nivel de inglés promedio
eng_cols = ['English Reading Lvl', 'English Spoken Lvl', 'English Writing Lvl']
df['English_Avg'] = df[eng_cols].mean(axis=1).round(2)

# Nivel de español promedio
spa_cols = ['Spanish Reading Lvl', 'Spanish Spoken Lvl', 'Spanish Writing Lvl']
df['Spanish_Avg'] = df[spa_cols].mean(axis=1).round(2)

# Limpiar 'Position vs Estimated Median' — convertir porcentaje a numérico
df['Position_vs_Median'] = df['Position vs Estimated Median'].astype(str).str.replace('%', '').str.strip()
df['Position_vs_Median'] = pd.to_numeric(df['Position_vs_Median'], errors='coerce')

# Rangos de edad
df['Age_Group'] = pd.cut(df['AGE'], bins=[18, 25, 30, 35, 40, 45, 50, 55, 60, 100],
                         labels=['18-25', '26-30', '31-35', '36-40', '41-45', '46-50', '51-55', '56-60', '60+'])

# Tenure buckets
df['Tenure_Group'] = pd.cut(df['Tenure_Years'], bins=[-1, 1, 3, 5, 8, 12, 50],
                            labels=['<1 año', '1-3 años', '3-5 años', '5-8 años', '8-12 años', '12+ años'])

print(f'Variables derivadas creadas. Nuevo shape: {df.shape}')
print(f'Distribución de Turnover: \n{df["Turnover"].value_counts()}')
df[['Employee Code', 'Turnover', 'Tenure_Years', 'Hire_Year', 'English_Avg', 'Age_Group', 'Tenure_Group']].head(10)

---
## 5. Análisis univariado — Variables categóricas

In [ ]:
# Variables categóricas clave
cat_cols = ['Gender', 'Marital Status', 'Area', 'Grouped Department Name', 
            'Type Location', 'Contract Type', 'Type', 'Non Exempt', 
            'Active Employee', 'Union', 'Probation Period Completed']

fig, axes = plt.subplots(4, 3, figsize=(20, 22))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    data = df[col].value_counts().head(10)
    bars = axes[i].barh(data.index.astype(str), data.values, color=sns.color_palette('viridis', len(data)))
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_xlabel('Frecuencia')
    for bar, val in zip(bars, data.values):
        axes[i].text(bar.get_width() + max(data.values)*0.01, bar.get_y() + bar.get_height()/2,
                    f'{val} ({val/len(df)*100:.1f}%)', va='center', fontsize=9)

# Ocultar ejes sobrantes
for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de Variables Categóricas', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Distribución detallada de departamentos
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

dept = df['Grouped Department Name'].value_counts()
axes[0].barh(dept.index, dept.values, color=sns.color_palette('Set2', len(dept)))
axes[0].set_title('Distribución por Departamento', fontweight='bold')
axes[0].set_xlabel('Cantidad de empleados')
for i, (val, name) in enumerate(zip(dept.values, dept.index)):
    axes[0].text(val + 10, i, f'{val} ({val/len(df)*100:.1f}%)', va='center', fontsize=9)

# Distribución de Job Level
jl = df['Job Level'].value_counts().sort_index()
axes[1].bar(jl.index.astype(str), jl.values, color=sns.color_palette('coolwarm', len(jl)))
axes[1].set_title('Distribución por Job Level', fontweight='bold')
axes[1].set_xlabel('Job Level')
axes[1].set_ylabel('Cantidad de empleados')
for x, val in zip(jl.index.astype(str), jl.values):
    axes[1].text(x, val + 10, str(val), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de razones de terminación (solo empleados que salieron)
terminated = df[df['Turnover'] == 1]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Termination Type
tt = terminated['Termination Type'].value_counts()
axes[0].pie(tt.values, labels=tt.index, autopct='%1.1f%%', startangle=90,
           colors=sns.color_palette('Set2', len(tt)))
axes[0].set_title('Tipo de Terminación', fontweight='bold')

# Top 10 Termination Reasons
tr = terminated['Termination Reason'].value_counts().head(10)
axes[1].barh(tr.index, tr.values, color=sns.color_palette('Reds_r', len(tr)))
axes[1].set_title('Top 10 Razones de Terminación', fontweight='bold')
axes[1].set_xlabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de ubicaciones
loc = df['Location Description'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(loc.index.astype(str), loc.values, color=sns.color_palette('viridis', len(loc)))
ax.set_title('Top 15 Ubicaciones', fontweight='bold')
ax.set_xlabel('Ubicación')
ax.set_ylabel('Cantidad de empleados')
plt.xticks(rotation=45, ha='right')
for bar, val in zip(bars, loc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 6. Análisis univariado — Variables numéricas

In [ ]:
num_cols = ['Hours/ Week', 'AGE', 'Standardized_Salary', 'Target Bonus %', 
            'Target Equity %', 'Exchange Rate', 'English_Avg', 'Spanish_Avg',
            'Tenure_Years', 'Position_vs_Median', 'Job Level']

fig, axes = plt.subplots(4, 3, figsize=(20, 22))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    axes[i].hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Media: {data.mean():.2f}')
    axes[i].axvline(data.median(), color='green', linestyle='-.', linewidth=1.5, label=f'Mediana: {data.median():.2f}')
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].legend(fontsize=8)

for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de Variables Numéricas (Histogramas)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots para detectar outliers
fig, axes = plt.subplots(4, 3, figsize=(20, 22))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    bp = axes[i].boxplot(data, vert=True, patch_artist=True, 
                         boxprops=dict(facecolor='lightblue', color='navy'),
                         medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'{col}', fontweight='bold')
    
    # Calcular outliers con IQR
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)).sum()
    axes[i].set_xlabel(f'Outliers: {outliers} ({outliers/len(data)*100:.1f}%)', fontsize=9)

for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots — Detección de Outliers', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla resumen de outliers
outlier_summary = []
for col in num_cols:
    data = df[col].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    n_outliers = ((data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)).sum()
    outlier_summary.append({
        'Variable': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Lím. Inferior': round(Q1 - 1.5*IQR, 2),
        'Lím. Superior': round(Q3 + 1.5*IQR, 2),
        'N Outliers': n_outliers,
        '% Outliers': round(n_outliers / len(data) * 100, 2),
        'Skewness': round(data.skew(), 2),
        'Kurtosis': round(data.kurtosis(), 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
print('RESUMEN DE OUTLIERS Y FORMA DE DISTRIBUCIÓN')
print('='*80)
outlier_df

---
## 7. Análisis bivariado

In [ ]:
# Turnover por variables categóricas clave
cat_vs_turnover = ['Gender', 'Marital Status', 'Grouped Department Name', 'Non Exempt',
                   'Type Location', 'Age_Group', 'Tenure_Group', 'Probation Period Completed']

fig, axes = plt.subplots(4, 2, figsize=(18, 24))
axes = axes.flatten()

for i, col in enumerate(cat_vs_turnover):
    ct = pd.crosstab(df[col], df['Turnover'], normalize='index') * 100
    ct.columns = ['Activo', 'Salió']
    ct.sort_values('Salió', ascending=True).plot(kind='barh', stacked=True, ax=axes[i],
                                                  color=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'Tasa de Rotación por {col}', fontweight='bold')
    axes[i].set_xlabel('Porcentaje (%)')
    axes[i].legend(loc='lower right')
    axes[i].set_xlim(0, 100)

plt.suptitle('Tasa de Rotación por Variables Categóricas', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla de tasas de rotación por categoría
print('TASAS DE ROTACIÓN POR CATEGORÍA')
print('='*70)
for col in cat_vs_turnover:
    ct = df.groupby(col)['Turnover'].agg(['mean', 'count'])
    ct.columns = ['Tasa Rotación', 'N']
    ct['Tasa Rotación'] = (ct['Tasa Rotación'] * 100).round(1)
    ct = ct.sort_values('Tasa Rotación', ascending=False)
    print(f'\n--- {col} ---')
    display(ct)

In [ ]:
# Salary vs Turnover
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Distribución de salario por turnover
for turnover_val, label, color in [(0, 'Activo', '#2ecc71'), (1, 'Salió', '#e74c3c')]:
    subset = df[df['Turnover'] == turnover_val]['Standardized_Salary'].dropna()
    axes[0, 0].hist(subset, bins=50, alpha=0.6, label=label, color=color, edgecolor='white')
axes[0, 0].set_title('Distribución de Salario Estandarizado por Turnover', fontweight='bold')
axes[0, 0].set_xlabel('Standardized Salary')
axes[0, 0].legend()

# Edad vs Turnover
for turnover_val, label, color in [(0, 'Activo', '#2ecc71'), (1, 'Salió', '#e74c3c')]:
    subset = df[df['Turnover'] == turnover_val]['AGE'].dropna()
    axes[0, 1].hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor='white')
axes[0, 1].set_title('Distribución de Edad por Turnover', fontweight='bold')
axes[0, 1].set_xlabel('Edad')
axes[0, 1].legend()

# Boxplot salario por turnover
df.boxplot(column='Standardized_Salary', by='Turnover', ax=axes[1, 0],
           patch_artist=True, boxprops=dict(facecolor='lightblue'))
axes[1, 0].set_title('Salario por Turnover', fontweight='bold')
axes[1, 0].set_xticklabels(['Activo', 'Salió'])
axes[1, 0].set_xlabel('')
plt.sca(axes[1, 0])
plt.title('Salario por Turnover', fontweight='bold')

# Boxplot edad por turnover
df.boxplot(column='AGE', by='Turnover', ax=axes[1, 1],
           patch_artist=True, boxprops=dict(facecolor='lightyellow'))
axes[1, 1].set_title('Edad por Turnover', fontweight='bold')
axes[1, 1].set_xticklabels(['Activo', 'Salió'])
axes[1, 1].set_xlabel('')
plt.sca(axes[1, 1])
plt.title('Edad por Turnover', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Nivel de inglés vs Turnover y Job Level
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Inglés vs Turnover
df.boxplot(column='English_Avg', by='Turnover', ax=axes[0], patch_artist=True)
axes[0].set_title('Nivel Inglés por Turnover', fontweight='bold')
axes[0].set_xticklabels(['Activo', 'Salió'])
axes[0].set_xlabel('')

# Salario por Job Level
df.boxplot(column='Standardized_Salary', by='Job Level', ax=axes[1], patch_artist=True)
axes[1].set_title('Salario por Job Level', fontweight='bold')
axes[1].set_xlabel('Job Level')

# Edad por departamento (top 6)
top_depts = df['Grouped Department Name'].value_counts().head(6).index
df_top = df[df['Grouped Department Name'].isin(top_depts)]
df_top.boxplot(column='AGE', by='Grouped Department Name', ax=axes[2], patch_artist=True)
axes[2].set_title('Edad por Departamento (Top 6)', fontweight='bold')
axes[2].set_xlabel('')
plt.sca(axes[2])
plt.xticks(rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Salario promedio por departamento y turnover
salary_dept = df.groupby(['Grouped Department Name', 'Turnover'])['Standardized_Salary'].mean().unstack()
salary_dept.columns = ['Activo', 'Salió']
salary_dept = salary_dept.sort_values('Activo', ascending=True)

fig, ax = plt.subplots(figsize=(14, 7))
salary_dept.plot(kind='barh', ax=ax, color=['#2ecc71', '#e74c3c'], width=0.7)
ax.set_title('Salario Estandarizado Promedio por Departamento y Estado', fontweight='bold')
ax.set_xlabel('Salario Estandarizado Promedio')
ax.legend(title='Estado')
plt.tight_layout()
plt.show()

---
## 8. Análisis de correlaciones

In [ ]:
# Seleccionar variables numéricas para correlación
corr_cols = ['Hours/ Week', 'AGE', 'Standardized_Salary', 'Target Bonus %', 
             'Target Equity %', 'English_Avg', 'Spanish_Avg', 'Job Level',
             'Tenure_Years', 'Position_vs_Median', 'Turnover', 'Exchange Rate',
             'English Reading Lvl', 'English Spoken Lvl', 'English Writing Lvl']

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación (Pearson)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlaciones más fuertes con Turnover
turnover_corr = corr_matrix['Turnover'].drop('Turnover').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in turnover_corr.values]
bars = ax.barh(turnover_corr.index, turnover_corr.values, color=colors)
ax.set_title('Correlación de variables con Turnover', fontweight='bold', fontsize=13)
ax.set_xlabel('Coeficiente de Correlación (Pearson)')
ax.axvline(x=0, color='black', linewidth=0.8)
for bar, val in zip(bars, turnover_corr.values):
    ax.text(val + 0.005 if val > 0 else val - 0.04, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop correlaciones con Turnover:')
print(turnover_corr.to_string())

In [ ]:
# Scatterplots de las relaciones más interesantes
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Salario vs Job Level coloreado por Turnover
scatter1 = axes[0, 0].scatter(df['Job Level'], df['Standardized_Salary'], 
                               c=df['Turnover'], cmap='RdYlGn_r', alpha=0.3, s=10)
axes[0, 0].set_xlabel('Job Level')
axes[0, 0].set_ylabel('Standardized Salary')
axes[0, 0].set_title('Salario vs Job Level (color=Turnover)', fontweight='bold')

# Edad vs Salario
scatter2 = axes[0, 1].scatter(df['AGE'], df['Standardized_Salary'],
                               c=df['Turnover'], cmap='RdYlGn_r', alpha=0.3, s=10)
axes[0, 1].set_xlabel('Edad')
axes[0, 1].set_ylabel('Standardized Salary')
axes[0, 1].set_title('Edad vs Salario (color=Turnover)', fontweight='bold')

# Tenure vs Salario
scatter3 = axes[1, 0].scatter(df['Tenure_Years'], df['Standardized_Salary'],
                               c=df['Turnover'], cmap='RdYlGn_r', alpha=0.3, s=10)
axes[1, 0].set_xlabel('Antigüedad (años)')
axes[1, 0].set_ylabel('Standardized Salary')
axes[1, 0].set_title('Antigüedad vs Salario (color=Turnover)', fontweight='bold')

# Nivel de inglés vs Salario
scatter4 = axes[1, 1].scatter(df['English_Avg'], df['Standardized_Salary'],
                               c=df['Turnover'], cmap='RdYlGn_r', alpha=0.3, s=10)
axes[1, 1].set_xlabel('Nivel Inglés Promedio')
axes[1, 1].set_ylabel('Standardized Salary')
axes[1, 1].set_title('Inglés vs Salario (color=Turnover)', fontweight='bold')

for ax in axes.flatten():
    ax.legend(*scatter1.legend_elements(), labels=['Activo', 'Salió'], loc='upper left', fontsize=8)

plt.suptitle('Scatterplots — Relaciones clave', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Análisis temporal

In [ ]:
# Contrataciones por año
hire_year = df['Hire_Year'].value_counts().sort_index()

# Terminaciones por año (solo empleados que salieron)
term_year = df[df['Turnover'] == 1]['Termination_Year'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].bar(hire_year.index.astype(int), hire_year.values, color='steelblue', edgecolor='white')
axes[0].set_title('Contrataciones por Año', fontweight='bold')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Cantidad')
for x, y in zip(hire_year.index, hire_year.values):
    axes[0].text(x, y + 5, str(y), ha='center', fontsize=8)

axes[1].bar(term_year.index.astype(int), term_year.values, color='#e74c3c', edgecolor='white')
axes[1].set_title('Terminaciones por Año', fontweight='bold')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Cantidad')
for x, y in zip(term_year.index, term_year.values):
    axes[1].text(x, y + 3, str(int(y)), ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Flujo neto: contrataciones - terminaciones por año
years = sorted(set(hire_year.index.astype(int)) | set(term_year.index.dropna().astype(int)))
flow = pd.DataFrame({'Año': years})
flow['Contrataciones'] = flow['Año'].map(dict(zip(hire_year.index.astype(int), hire_year.values))).fillna(0)
flow['Terminaciones'] = flow['Año'].map(dict(zip(term_year.index.dropna().astype(int), term_year.values))).fillna(0)
flow['Flujo Neto'] = flow['Contrataciones'] - flow['Terminaciones']

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(flow['Año'], flow['Contrataciones'], 'o-', color='steelblue', linewidth=2, label='Contrataciones')
ax.plot(flow['Año'], flow['Terminaciones'], 'o-', color='#e74c3c', linewidth=2, label='Terminaciones')
ax.bar(flow['Año'], flow['Flujo Neto'], alpha=0.3,
       color=['green' if x >= 0 else 'red' for x in flow['Flujo Neto']], label='Flujo Neto')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_title('Flujo de Personal por Año', fontweight='bold', fontsize=13)
ax.set_xlabel('Año')
ax.set_ylabel('Cantidad de empleados')
ax.legend()
plt.tight_layout()
plt.show()

print('Flujo de personal:')
flow.set_index('Año')

In [ ]:
# Distribución de antigüedad
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Por grupo de antigüedad
tenure_g = df['Tenure_Group'].value_counts().sort_index()
axes[0].bar(tenure_g.index.astype(str), tenure_g.values, color=sns.color_palette('Blues_d', len(tenure_g)))
axes[0].set_title('Distribución por Grupo de Antigüedad', fontweight='bold')
axes[0].set_xlabel('Antigüedad')
axes[0].set_ylabel('Empleados')
for i, (x, y) in enumerate(zip(tenure_g.index.astype(str), tenure_g.values)):
    axes[0].text(i, y + 10, str(y), ha='center')

# Antigüedad por turnover
df.boxplot(column='Tenure_Years', by='Turnover', ax=axes[1], patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Antigüedad por Estado del Empleado', fontweight='bold')
axes[1].set_xticklabels(['Activo', 'Salió'])
axes[1].set_xlabel('')
axes[1].set_ylabel('Años de antigüedad')

plt.tight_layout()
plt.show()

---
## 10. Análisis de rotación (Turnover) en profundidad

In [ ]:
# Tasa de rotación global
total = len(df)
activos = df['Turnover'].value_counts().get(0, 0)
salieron = df['Turnover'].value_counts().get(1, 0)

print('RESUMEN DE ROTACIÓN')
print('='*50)
print(f'Total empleados en el dataset:  {total}')
print(f'Empleados activos:              {activos} ({activos/total*100:.1f}%)')
print(f'Empleados que salieron:         {salieron} ({salieron/total*100:.1f}%)')
print(f'Tasa de rotación global:        {salieron/total*100:.1f}%')

# Por tipo de terminación
print(f'\nPor tipo de terminación:')
print('-'*50)
tt_counts = terminated['Termination Type'].value_counts()
for tipo, count in tt_counts.items():
    print(f'  {tipo}: {count} ({count/salieron*100:.1f}% de las salidas)')

In [ ]:
# Perfil comparativo: Activos vs. Salieron
comparison_cols = ['AGE', 'Standardized_Salary', 'Tenure_Years', 'Job Level',
                   'Hours/ Week', 'Target Bonus %', 'English_Avg', 'Position_vs_Median']

comparison = df.groupby('Turnover')[comparison_cols].agg(['mean', 'median', 'std']).round(3)
comparison.index = ['Activo', 'Salió']

print('PERFIL COMPARATIVO: ACTIVOS vs. SALIERON')
print('='*80)
comparison

In [ ]:
# Tasa de rotación por Job Level
jl_turnover = df.groupby('Job Level').agg(
    Total=('Turnover', 'count'),
    Salieron=('Turnover', 'sum'),
    Tasa_Rotacion=('Turnover', 'mean')
).round(3)
jl_turnover['Tasa_Rotacion'] = (jl_turnover['Tasa_Rotacion'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(jl_turnover.index.astype(str), jl_turnover['Tasa_Rotacion'], 
           color=sns.color_palette('YlOrRd', len(jl_turnover)))
axes[0].set_title('Tasa de Rotación por Job Level', fontweight='bold')
axes[0].set_xlabel('Job Level')
axes[0].set_ylabel('Tasa de Rotación (%)')
for x, y in zip(jl_turnover.index.astype(str), jl_turnover['Tasa_Rotacion']):
    axes[0].text(x, y + 0.5, f'{y}%', ha='center', fontsize=9)

# Rotación voluntaria vs involuntaria por Job Level
vol_inv = terminated.groupby(['Job Level', 'Termination Type']).size().unstack(fill_value=0)
vol_inv_pct = vol_inv.div(vol_inv.sum(axis=1), axis=0) * 100
vol_inv_pct.plot(kind='bar', stacked=True, ax=axes[1], color=['#e74c3c', '#f39c12', '#3498db'])
axes[1].set_title('Tipo de Terminación por Job Level', fontweight='bold')
axes[1].set_xlabel('Job Level')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].legend(title='Tipo', fontsize=8)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print('Detalle:')
jl_turnover

In [ ]:
# Tasa de rotación por ubicación
loc_turnover = df.groupby('Location Description').agg(
    Total=('Turnover', 'count'),
    Salieron=('Turnover', 'sum'),
    Tasa_Rotacion=('Turnover', 'mean')
).round(3)
loc_turnover['Tasa_Rotacion'] = (loc_turnover['Tasa_Rotacion'] * 100).round(1)
loc_turnover = loc_turnover.sort_values('Tasa_Rotacion', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
colors = sns.color_palette('YlOrRd', len(loc_turnover))
ax.barh(loc_turnover.index, loc_turnover['Tasa_Rotacion'], color=colors)
ax.set_title('Tasa de Rotación por Ubicación', fontweight='bold')
ax.set_xlabel('Tasa de Rotación (%)')
for i, (val, total) in enumerate(zip(loc_turnover['Tasa_Rotacion'], loc_turnover['Total'])):
    ax.text(val + 0.3, i, f'{val}% (n={total})', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de rotación temprana (primeros 2 años)
early_turnover = terminated[terminated['Tenure_Years'] <= 2]
print('ROTACIÓN TEMPRANA (≤ 2 años de antigüedad)')
print('='*60)
print(f'Empleados que salieron en los primeros 2 años: {len(early_turnover)} ({len(early_turnover)/salieron*100:.1f}% de todas las salidas)')
print(f'\nRazones principales de salida temprana:')
print(early_turnover['Termination Reason'].value_counts().head(10).to_string())
print(f'\nTipo de terminación en salidas tempranas:')
print(early_turnover['Termination Type'].value_counts().to_string())
print(f'\nDepartamentos con más salidas tempranas:')
print(early_turnover['Grouped Department Name'].value_counts().head(6).to_string())

---
## 11. Resumen de hallazgos clave

In [ ]:
print('='*70)
print('      RESUMEN EJECUTIVO — HALLAZGOS CLAVE DEL EDA')
print('='*70)

print(f'''
1. DIMENSIÓN DEL DATASET
   - {df.shape[0]} empleados, {df.shape[1]} variables
   - Employee Codes únicos: {df['Employee Code'].nunique()}

2. TASA DE ROTACIÓN
   - Tasa global: {salieron/total*100:.1f}%
   - Activos: {activos} | Salieron: {salieron}
   - Voluntarias: {terminated[terminated['Termination Type']=='Voluntary'].shape[0]} | Involuntarias: {terminated[terminated['Termination Type']=='Involuntary'].shape[0]}

3. PERFIL DEMOGRÁFICO
   - Edad promedio: {df['AGE'].mean():.1f} años (rango {df['AGE'].min()}-{df['AGE'].max()})
   - Género: {df['Gender'].value_counts().to_dict()}
   - Antigüedad promedio: {df['Tenure_Years'].mean():.1f} años

4. VARIABLES CLAVE ASOCIADAS AL TURNOVER
   (Correlaciones de Pearson con Turnover)
''')

for var, corr_val in turnover_corr.head(6).items():
    direction = '↑ más rotación' if corr_val > 0 else '↓ menos rotación'
    print(f'   - {var}: r={corr_val:.3f} ({direction})')

print(f'''
5. CALIDAD DE DATOS
   - Filas duplicadas: {dup_total}
   - Columnas con >10% nulos: {len(null_analysis[null_analysis["% Nulos"] > 10])}
   - Fechas inválidas (1900): presentes en Termination Date para activos

6. ALERTAS
   - Verificar registros con Employee Code duplicado
   - Las fechas de terminación "1900" son placeholders para empleados activos
   - La columna 'Position vs Estimated Median' tiene formato mixto (% como texto)
   - Algunas razones de terminación contienen texto inconsistente (e.g., "DO NOT USE")
''')
print('='*70)
print('Fin del Análisis Exploratorio de Datos')
print('='*70)